# OpenCoesione: ratio pagamenti / finanziamento UE per macro-tema

Supporto analitico minimo per una prima discussion `Analisi`.

- **Obiettivo**: verificare se esiste un macro-tema che si discosta in modo riconoscibile nel ratio tra pagamenti e finanziamento UE osservato.
- **Dati**: mart `mart_regione_tema` dal filone `opencoesione-pagamenti-ue-2014-2020`.
- **Perimetro**: aggregazione nazionale per 11 macro-temi.
- **Nota**: i casi con ratio > 1 non vanno interpretati senza una verifica metodologica aggiuntiva.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.options.display.float_format = '{:,.3f}'.format

mart_path = Path('../../../dataset-incubator/out/data/mart/opencoesione_pagamenti_ue_2014_2020/2020/mart_regione_tema.parquet')

if not mart_path.exists():
    raise FileNotFoundError(f'Mart non trovato: {mart_path.resolve()}')

con = duckdb.connect()
mart_path

WindowsPath('../../../dataset-incubator/out/data/mart/opencoesione_pagamenti_ue_2014_2020/2020/mart_regione_tema.parquet')

In [2]:
temi_ratio = con.execute(
    """
    select
        tema,
        sum(finanz_ue_tot) as finanz_ue_tot,
        sum(tot_pagamenti_tot) as tot_pagamenti_tot,
        sum(tot_pagamenti_tot) / sum(finanz_ue_tot) as ratio_spesa
    from read_parquet($path)
    group by 1
    order by ratio_spesa asc, finanz_ue_tot desc
    """,
    {'path': str(mart_path)},
).fetchdf()

temi_ratio

,tema,finanz_ue_tot,tot_pagamenti_tot,ratio_spesa
0,Ambiente,"3,726,336,666.590","3,432,584,815.130",0.921
1,Trasporti e mobilità,"4,898,241,403.300","4,878,213,925.810",0.996
2,Energia,"1,959,581,296.550","2,063,714,204.390",1.053
3,Cultura e turismo,"1,111,716,517.770","1,237,954,962.920",1.114
4,Reti e servizi digitali,"3,250,695,011.560","3,646,783,022.560",1.122
5,Competitività delle imprese,"6,190,665,581.350","7,170,677,228.220",1.158
6,Capacità amministrativa,"1,579,039,729.540","1,831,368,899.230",1.160
7,Occupazione e lavoro,"4,187,285,908.670","5,176,845,622.070",1.236
8,Istruzione e formazione,"4,890,523,775.150","6,082,090,066.480",1.244
9,Inclusione sociale e salute,"4,662,045,092.020","5,878,393,003.030",1.261


In [3]:
temi_ratio[['tema', 'ratio_spesa']]


,tema,ratio_spesa
0,Ambiente,0.921
1,Trasporti e mobilità,0.996
2,Energia,1.053
3,Cultura e turismo,1.114
4,Reti e servizi digitali,1.122
5,Competitività delle imprese,1.158
6,Capacità amministrativa,1.160
7,Occupazione e lavoro,1.236
8,Istruzione e formazione,1.244
9,Inclusione sociale e salute,1.261


### Caveat

- **Ratio > 1**: non va letto automaticamente come spesa oltre il finanziamento assegnato; serve una verifica metodologica sul perimetro della quota UE osservata.
- **Efficacia**: il ratio misura la traiettoria dei pagamenti, non la qualita' o l'impatto degli interventi.
- **Aggregazione**: ogni macro-tema contiene progetti molto diversi; il dato nazionale non dice ancora dove si concentra l'anomalia osservata.
- **Perimetro**: questa e' una prima vista tematica nazionale, non una lettura completa del ciclo.